#### Importações 

In [25]:
from seleniumbase import SB
import pandas as pd
from datetime import datetime
import json

URL = "https://www.saucedemo.com/"
username = "standard_user"
password = "secret_sauce"

first_name = "Trainee"
last_name = "PiJunior"
zip_code = "31270-901"

### Funções de utilidade

In [26]:
def log(message):
    now = datetime.now().strftime("%H:%M:%S")
    print(f"[{now}] {message}")

In [27]:
def dict_to_json(dict,file_name):
    with open(file_name,"w",encoding="utf-8") as file:
        json.dump(dict, file, ensure_ascii=False, indent=4)

#### 1. Login Automatizado

In [28]:
def login(sb, username, password):
    # digitando nome e senha nos inputs
    sb.type('[name="user-name"]', username)
    sb.type('[name="password"]', password)

    log("Realizando login")
    sb.click('#login-button')

#### 2. Coleta de Dados

In [32]:
data = []

with SB() as sb:
    log(f"Abrindo site {URL}")
    sb.open(URL) # abrindo url

    # verificando se o html foi carregado
    if sb.get_text(".login_logo") == "Swag Labs":
        log("Site aberto com sucesso!")
    else: 
        log("Deu alguma merda")

    login(sb,username,password)

    # esperando os itens do inventário aparecer
    sb.wait_for_element_visible(
        '.inventory_item',
        timeout=10
    )
    log("Login Realizado com Sucesso!")

    log("Extraindo dados dos itens")
    inventory_items = sb.find_elements(".inventory_item")
    for item in inventory_items:
        # extraindo dados da página inicial
        item_name = item.find_element("css selector",".inventory_item_name").text
        item_desc = item.find_element("css selector",".inventory_item_desc").text
        item_price = item.find_element("css selector",".inventory_item_price").text
        
        item_dict = {
            "item_name":item_name,
            "item_desc":item_desc,
            "item_price":item_price,
        }

        data.append(item_dict)
    
    log("Dados extraídos com sucesso")

    # transformando dict em data frame
    df_produtos = pd.DataFrame(data)


[15:27:22] Abrindo site https://www.saucedemo.com/
[15:27:23] Site aberto com sucesso!
[15:27:24] Realizando login
[15:27:24] Login Realizado com Sucesso!
[15:27:24] Extraindo dados dos itens
[15:27:25] Dados extraídos com sucesso


### 3. Automação de Compra

In [35]:
payment_information = {}

with SB() as sb:
    log(f"Abrindo site {URL}")
    sb.open(URL) # abrindo url

    # verificando se o html foi carregado
    if sb.get_text(".login_logo") == "Swag Labs":
        log("Site aberto com sucesso!")
    else: 
        log("Deu alguma merda")

    login(sb,username,password)

    # adicionando produtos ao carrinho
    log("Adicionando produtos ao carrinho")
    buttons = sb.find_elements("button.btn_inventory")

    for button in buttons:
        button.click()

    log("Todos os produtos foram adicionados ao carrinho!")

    # checkout
    log("Abrindo carrinho")
    sb.click(".shopping_cart_link")

    sb.wait_for_element_visible(
        "#checkout", 
        timeout=10)
    sb.click("#checkout")

    sb.wait_for_element_visible(
        "#first-name", 
        timeout=10)

    sb.type("#first-name", first_name)
    sb.type("#last-name", last_name)
    sb.type("#postal-code", zip_code)

    sb.click("#continue")

    log("Salvando informações da compra")
    payment = sb.get_text('[data-test="payment-info-value"]')
    shipping = sb.get_text('[data-test="shipping-info-value"]')
    subtotal = sb.get_text('[data-test="subtotal-label"]')
    tax = sb.get_text('[data-test="tax-label"]')
    total = sb.get_text('[data-test="total-label"]')

    payment_information = {
        "Meio de Pagamento": payment,
        "Forma de Entrega": shipping,
        "Preço Total": {
            "Preço dos itens": subtotal,
            "Taxas": tax,
            "Total": total
        }
    }

    # finalizando a compra
    sb.click("#finish")

    sb.wait_for_element_visible(
        ".complete-header",
        timeout=10
    )

    confirmation_message = sb.get_text(
        ".complete-header"
    )

    print()
    print(confirmation_message)


[15:28:56] Abrindo site https://www.saucedemo.com/
[15:28:57] Site aberto com sucesso!
[15:28:58] Realizando login
[15:28:58] Adicionando produtos ao carrinho
[15:28:59] Todos os produtos foram adicionados ao carrinho!
[15:28:59] Abrindo carrinho
[15:29:00] Salvando informações da compra

Thank you for your order!


#### Exportando os dados

In [36]:
df_produtos.to_csv("produtos.csv", index=False)

dict_to_json(payment_information,"informacoes_pagamento.json")